# Lab Day 1 — Notebook (PSEUDO-CODE)

> **Đây là khung pseudo-code.** Mọi ô có `# TODO` là phần bạn phải tự viết. Hãy giữ cấu trúc mục (Part 0–4) và
> viết dự đoán/nhận xét bằng chữ của bạn. Đọc `README.md`, `GUIDE.md`, `RUBRIC.md` trước.
>
> **Quy trình làm bài**
> 1. `git pull` repo, tạo thư mục nộp: copy cả thư mục `code/` vào `submission_<MSSV>/code/`.
> 2. Mở `submission_<MSSV>/code/lab.ipynb` và làm việc **tại đó** (nên `REPO_ROOT = "../.."` đúng và ảnh/kết quả ghi vào `submission_<MSSV>/figures`, `results`).
> 3. Ở Colab: chạy ô clone repo bên dưới, rồi làm tương tự (đặt `REPO_ROOT` cho đúng).
> 4. Cuối cùng: `Restart & Run All` phải chạy hết không lỗi và giữ nguyên output.

In [ ]:
# ===== Cấu hình đường dẫn và môi trường =====
import os, sys, json, time, subprocess
from pathlib import Path
import numpy as np, torch

# Xử lý đường dẫn linh hoạt (hỗ trợ cả chạy trên Colab và chạy local)
if 'google.colab' in sys.modules:
    REPO_NAME = 'K4-Track4-Day1-DuongVanThanh-2A202602368-Neuralnetwork'
    if not os.path.exists(f'/content/{REPO_NAME}'):
        subprocess.run(['git', 'clone', f'https://github.com/JJayzdev/{REPO_NAME}.git'], check=True)
    REPO_ROOT = f'/content/{REPO_NAME}'
    OUT_DIR = f'{REPO_ROOT}/submission_2A202602368'
    os.chdir(f'{OUT_DIR}/code')
    if f'{OUT_DIR}/code' not in sys.path:
        sys.path.insert(0, f'{OUT_DIR}/code')
else:
    REPO_ROOT = '../..'     # thư mục gốc repo, tính từ submission_<MSSV>/code/
    OUT_DIR   = '..'        # nơi ghi figures/, results/, experiments.xlsx, predictions_eval.csv
    if '.' not in sys.path:
        sys.path.insert(0, '.')

# Thiết bị: Ưu tiên CUDA nếu có GPU, ngược lại CPU (trên Mac có thể mps)
device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() else 'cpu'))
print(f'PyTorch version: {torch.__version__}')
print(f'Thiết bị đang sử dụng: {device}')
if device.type == 'cuda':
    print(f'Tên GPU: {torch.cuda.get_device_name(0)}')

# Đảm bảo các thư mục đầu ra tồn tại
os.makedirs(f'{OUT_DIR}/figures', exist_ok=True)
os.makedirs(f'{OUT_DIR}/results', exist_ok=True)

from data import prepare_data, iterate_batches
from model import MLP, EXPECTED_PARAMS, count_params, init_weights, activation_stats
from optimizer import build_optimizer, clip_gradients
from train import DEFAULT_CFG, set_seed, evaluate, predict, run_experiment, final_eval
from plots import plot_run, plot_compare
from results_table import save_result, load_results, to_row, write_xlsx


## Part 0 — Dữ liệu
Chia sẵn bằng metadata: chạy `scripts/split_data.py` từ thư mục gốc repo (tạo `data/processed/train.npz`, `eval.npz`).
Sau đó tách validation từ train, chuẩn hoá bằng thống kê của train, đưa lên device.

In [ ]:
# 1. Chạy split_data.py nếu chưa có train.npz và eval.npz
processed_dir = os.path.join(REPO_ROOT, 'data', 'processed')
train_npz = os.path.join(processed_dir, 'train.npz')
eval_npz = os.path.join(processed_dir, 'eval.npz')

if not (os.path.exists(train_npz) and os.path.exists(eval_npz)):
    print('Đang chạy scripts/split_data.py để tạo dữ liệu ban đầu...')
    subprocess.run([sys.executable, 'scripts/split_data.py'], cwd=REPO_ROOT, check=True)
else:
    print('Dữ liệu train.npz và eval.npz đã sẵn sàng.')

# 2. Nạp dữ liệu, tách validation 20% stratified seed 42, chuẩn hóa và chuyển lên device
data = prepare_data(device=device, val_fraction=0.2, seed=42, processed_dir=processed_dir)

# 3. Trích xuất các tensor và kiểm tra kích thước / kiểu dữ liệu
X_tr, y_tr = data['X_tr'], data['y_tr']
X_val, y_val = data['X_val'], data['y_val']
X_eval, y_eval = data['X_eval'], data['y_eval']

print('--- KIỂM TRA ĐẦU RA PART 0 ---')
print(f'Kích thước X_tr   : {X_tr.shape}, kiểu: {X_tr.dtype}')
print(f'Kích thước y_tr   : {y_tr.shape}, kiểu: {y_tr.dtype}')
print(f'Kích thước X_val  : {X_val.shape}, kiểu: {X_val.dtype}')
print(f'Kích thước y_val  : {y_val.shape}, kiểu: {y_val.dtype}')
print(f'Kích thước X_eval : {X_eval.shape}, kiểu: {X_eval.dtype}')
print(f'Kích thước y_eval : {y_eval.shape}, kiểu: {y_eval.dtype}')
print(f'Số lượng eval_row_id: {len(data["eval_row_id"])}')

# Kiểm tra shape theo tiêu chuẩn quy định trong GUIDE
assert X_tr.shape == (371847, 54), f'Kỳ vọng 371847 mẫu train nhưng nhận {X_tr.shape[0]}'
assert X_val.shape == (92962, 54), f'Kỳ vọng 92962 mẫu val nhưng nhận {X_val.shape[0]}'
assert X_eval.shape == (116203, 54), f'Kỳ vọng 116203 mẫu eval nhưng nhận {X_eval.shape[0]}'
assert X_tr.dtype == torch.float32 and y_tr.dtype == torch.int64
assert X_val.dtype == torch.float32 and y_val.dtype == torch.int64
assert X_eval.dtype == torch.float32 and y_eval.dtype == torch.int64
assert len(data['eval_row_id']) == 116203

# 4. Kiểm tra chuẩn hóa 10 cột số liên tục trên X_tr (mean ~ 0, std ~ 1)
numeric_means = X_tr[:, :10].mean(dim=0).cpu().numpy()
numeric_stds = X_tr[:, :10].std(dim=0).cpu().numpy()
print('\n--- KIỂM TRA THỐNG KÊ 10 CỘT LIÊN TỤC TRÊN X_tr ---')
print(f'Mean của 10 cột liên tục:\n{np.round(numeric_means, 4)}')
print(f'Std của 10 cột liên tục:\n{np.round(numeric_stds, 4)}')
assert np.allclose(numeric_means, 0.0, atol=1e-2), 'Mean của 10 cột trên train chưa chuẩn hóa về 0!'
assert np.allclose(numeric_stds, 1.0, atol=1e-2), 'Std của 10 cột trên train chưa chuẩn hóa về 1!'
print('=> Chuẩn hóa 10 cột số liên tục đạt yêu cầu (mean ≈ 0, std ≈ 1). 44 cột nhị phân giữ nguyên.')

# 5. Đo accuracy của chiến lược luôn đoán lớp đa số trên tập validation
val_counts = torch.bincount(y_val, minlength=7)
maj_class = torch.argmax(val_counts).item()
majority_acc = (y_val == maj_class).float().mean().item()
print(f'\n--- MỐC SÀN DỰ ĐOÁN ĐA SỐ (MAJORITY BASELINE) ---')
print(f'Lớp đa số trên validation: Lớp {maj_class} ({val_counts[maj_class].item()}/{len(y_val)} mẫu, chiếm {val_counts[maj_class].item()/len(y_val)*100:.2f}%)')
print(f'Accuracy đoán lớp đa số trên val: {majority_acc:.4f} (Mốc sàn tối thiểu mô hình phải vượt ≈ 0.4876)')


### Nhận xét Part 0 (Tự viết):
1. **Kích thước và phân chia tập dữ liệu**:
   - Dữ liệu gốc gồm 581 012 mẫu được chia thành `train` (464 809 mẫu) và `eval` (116 203 mẫu) cố định theo `split_metadata.csv`.
   - Tách validation 20% phân tầng theo nhãn từ tập train (seed 42) thu được: **371 847 mẫu train** và **92 962 mẫu validation**.
   - Mảng `eval_row_id` được bảo toàn nguyên vẹn đúng thứ tự (116 203 mẫu) để đảm bảo sinh file dự đoán `predictions_eval.csv` chính xác ở Part 4.
2. **Cơ chế chuẩn hóa và phòng tránh rò rỉ dữ liệu (Data Leakage)**:
   - Chỉ chuẩn hóa 10 đặc trưng số liên tục đầu tiên bằng z-score ($z = (x - \mu)/\sigma$); 44 đặc trưng nhị phân One-hot phía sau được giữ nguyên.
   - Thông số $(\mu, \sigma)$ **bắt buộc chỉ được tính trên tập train rút gọn** (sau khi tách validation) và áp dụng cùng thông số này cho validation và eval.
   - **Lý do không được dùng validation hay eval để tính mean/std**: Nếu tính mean/std trên toàn bộ tập dữ liệu hoặc trên eval, mô hình sẽ tiếp cận trước phân phối thống kê của tập đánh giá (data leakage), làm sai lệch tính khách quan và khả năng tổng quát hóa thực tế của mô hình.
3. **Chiến lược đoán lớp đa số (Majority Baseline)**:
   - Do mất cân bằng lớp (lớp 1 chiếm 48.76%), nếu mô hình luôn dự đoán lớp 1 thì đạt Accuracy $\approx 0.4876$ trên validation, nhưng Macro-F1 chỉ $\approx 0.094$.
   - Mọi mô hình huấn luyện sau này bắt buộc phải vượt xa mốc sàn Accuracy này, và thước đo đánh giá chất lượng cốt lõi của bài toán là **Macro-F1**.

## Part 1 — Model và kiểm tra "sức khoẻ" ban đầu
Quy định: `M-base` = `54 → 256 → 128 → 7`, **47 879 tham số**, logits `(B, 7)`, không softmax trong model.

In [ ]:
# TODO 1: model = MLP(hidden=(256, 128), dropout=0.0, init="he").to(device); assert count_params(model) == EXPECTED_PARAMS[(256,128)]
# TODO 2: cho một lô ngẫu nhiên (8, 54) chạy qua; in shape; assert logits.shape == (8, 7)
# TODO 3: loss bước 0 trên val (eval mode) ≈ ln 7 ≈ 1.946  -> in giá trị đo và nhận xét
# TODO 4: quá khớp 20 mẫu (tắt dropout, vài trăm bước) -> loss về gần 0; vẽ đường cong loss
# TODO 5: sau một lần backward, in grad norm của từng tham số; assert tất cả khác None và khác 0

**Nhận xét Part 1 (tự viết):** loss bước 0 đo được = ___, so với ln 7 = 1.946 vì sao ...; quá khớp 20 mẫu: ...

## Part 2 — Pipeline và baseline
`run_experiment(cfg, data)` nhận một dict cấu hình và trả về lịch sử + tóm tắt. Baseline: SGD+momentum 0.9, CE, He init, batch 512, 20 epoch.

In [ ]:
# TODO: chọn lr cho baseline bằng VAL (chạy vài giá trị, ví dụ quanh 0.01 -> 0.1); ghi bảng lr -> val_macro_f1
# TODO: cfg = {**DEFAULT_CFG, "lr": <lr đã chọn>}
# TODO: chạy baseline với >= 2 seed (khuyến khích 3): result = run_experiment({**cfg, "seed": s, "exp_id": f"base-s{s}"}, data)
# TODO: với mỗi result: save_result(result, f"{OUT_DIR}/results"); plot_run(result, f"{OUT_DIR}/figures/{exp_id}.png")
# TODO: tính trung bình ± độ lệch chuẩn (val_macro_f1, val_acc) qua các seed -> ngưỡng nhiễu 2σ

**Nhận xét baseline (tự viết):** hình dạng đường cong (còn giảm? bắt đầu quá khớp?), val_acc so với mốc "đoán đa số", độ nhiễu giữa các seed ...

## Part 3 — Thí nghiệm (tự chọn; menu trong GUIDE, Part 3)
Mỗi thí nghiệm: **dự đoán trước** → đổi **một** yếu tố → chạy → **đối chiếu**. Lặp lại khối dưới cho từng thí nghiệm bạn chọn.
Chủ đề: loss · optimizer · hparam · dropout · clipping · amp · init.

### Thí nghiệm `<exp_id>` — chủ đề `<nhóm>`
**Dự đoán trước (viết TRƯỚC khi chạy):** ...

In [ ]:
# Mẫu một thí nghiệm. Lặp lại / bọc trong vòng for qua danh sách cfg bạn tự thiết kế.
# TODO: exp_cfg = {**cfg, "exp_id": "...", "group": "...", "description": "...", <chỉ đổi MỘT yếu tố>}
# TODO: result = run_experiment(exp_cfg, data)
# TODO: save_result(...); plot_run(...)   # mỗi thí nghiệm một ảnh figures/<exp_id>.png
# TODO: in summary (best_val_loss, val_acc, val_macro_f1, ...) và so với baseline + ngưỡng nhiễu 2σ

**Đối chiếu sau khi chạy:** khớp / khác dự đoán? vì sao (cơ chế)? chênh lệch có vượt 2σ không? ...

In [ ]:
# TODO: ảnh chồng theo nhóm: plot_compare(list_of_results_in_group, "val_loss", f"{OUT_DIR}/figures/compare_<nhóm>.png")

## Part 4 — Đánh giá cuối trên eval, bảng và nộp bài
**Cách đánh giá:** chọn cấu hình CHỈ bằng val → chạy final model → dự đoán trên **toàn bộ** tập eval → ghi `predictions_eval.csv`
→ `python scripts/evaluate.py --pred ...` tính accuracy và **macro-F1** (chỉ số chính). Chỉ làm bước này cho baseline và cấu hình cuối cùng.

In [ ]:
# TODO: chọn cấu hình cuối cùng theo VAL (có thể là baseline nếu bạn không kết hợp thêm gì)
# TODO: final_eval(cfg_final, result_final, data, f"{OUT_DIR}/predictions_eval.csv")
# TODO: chạy: python scripts/evaluate.py --pred <OUT_DIR>/predictions_eval.csv --out <OUT_DIR>/eval_result.json (từ REPO_ROOT)
# TODO: đọc eval_result.json: ghi accuracy, macro_f1 vào bảng và báo cáo; làm tương tự cho baseline (predictions riêng, không nộp nếu không cần)

In [ ]:
# TODO: phân tích lỗi trên eval: bảng precision/recall/F1 theo lớp (từ eval_result.json) và ma trận nhầm lẫn;
#       lớp nào khó nhất? vì sao (số mẫu ít? nhầm với lớp nào?)  -> viết nhận xét ở ô markdown bên dưới

**Phân tích lỗi (tự viết):** ...

In [ ]:
# TODO: results = load_results(f"{OUT_DIR}/results"); rows = [to_row(r, ...) for r in results]
# TODO: write_xlsx(rows, f"{REPO_ROOT}/templates/experiment_table_template.xlsx", f"{OUT_DIR}/experiments.xlsx")
# TODO: bổ sung sheet Seeds (exp_id các lần chạy baseline) và nhận xét ở sheet Summary